## Intialising Function and Importing Modules

In [1]:
from trainModel import trainModel
from Dataset import ModelDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, accuracy_score
import pandas as pd
import os
from generateSplits import generateSplits
from torch.utils.data import DataLoader
import torch,gc
import numpy as np
from typing import Literal, Callable, Iterator

In [2]:
DATASET_PATH = r"E:\SRP\SRP-2025-Project\ISPY2_T0_T3_DCE_npz"

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset_df = pd.read_excel(r"e:\SRP\ISPY2-Data-Collector\ISPY2-Imaging-Cohort-1-Clinical-Data.xlsx")
dataset_df = dataset_df.set_index("Patient_ID",drop=True)
dataset_df = dataset_df.loc[dataset_df.index.isin([int(os.path.splitext(fname)[0].replace("ISPY2-","")) for fname in os.listdir(DATASET_PATH)]),["HR","HER2","pCR"]]

train_df, test_df = generateSplits(dataset_df,0.2,seed=2008)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=2008)

In [3]:
best_params = {'lr': 0.010203264127576701, 'weight_decay': 0.004617603477095672, 'batch_size': 8, 'optimiser_name': 'Adam'}

In [4]:
def evaluate_roc_auc(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],testKwargs:dict):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols,**testKwargs)
            else:
                logits = model(T0_volumes,T3_volumes,mols,**testKwargs)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return roc_auc_score(y_true,y_score)

In [5]:
def get_scores(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],mode:Literal["preNac","both"]="preNac"):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols,mode)
            else:
                logits = model(T0_volumes,T3_volumes,mols,mode)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return y_true,y_score

In [6]:
def score_to_pred(y_true,y_score):
    fpr,tpr,thresholds = roc_curve(y_true,y_score)
    youden_index = tpr-fpr
    best_idx = np.argmax(youden_index)
    best_threshold = thresholds[best_idx]
    y_pred = np.array(y_score)>=best_threshold
    return y_pred

In [7]:
def evaluate_model(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],mode:Literal["preNac","both"]):
    y_true,y_score = get_scores(model,
                                test_loader,
                                combine_timepoints,
                                out_features,
                                mode)
    y_pred = score_to_pred(y_true,y_score)
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()
    return pd.Series({"ROC AUC":roc_auc_score(y_true,y_score),
                      "Average Precision Score":average_precision_score(y_true,y_score),
                      "Sensitivity / Recall / TPR":(tp/(tp+fn)),
                      "Specificity / TNR":(tn/(tn+fp)),
                      "PPV / Precision":(tp/(tp+fp)),
                      "NPV":(tn/(tn+fn)),
                      "Accuracy":accuracy_score(y_true,y_pred)})

In [8]:
def four_fold_cv_train(Model_class:type[torch.nn.Module],
                       optimiser_class:Callable[[Iterator[torch.nn.Parameter]],torch.optim.Optimizer],
                       class_samples:dict,
                       num_epochs:int,
                       batch_size:int,
                       combine_timepoints:bool,
                       output_features:Literal[1,2],
                       loss_fn:Callable,
                       score_fn:Callable,
                       score_name:str,
                       probs_fn:Callable[[torch.nn.Module,DataLoader,bool,Literal[1,2],Literal["preNac","both"]],tuple],
                       modelKwargs:dict,
                       trainKwargs:dict,
                       testKwargs:dict):
    '''Train model using four-fold cross validation
    
    Parameters
    ----------
    Model_Class : Class to intialise the model
    
    optimiser_class : Class to intialise optimiser
    
    class_samples : Proportion of augmentation for positive and negatives
    
    num_epochs : Number of Epochs
    
    batch_size : Batch Size
    
    combine_timepoints : Whether to combine T0 and T3 into one volume
    
    output_features : Whether the output logits of the model is 1 or 2
    
    loss_fn : Used to calculate loss when training
    
    score_fn : Score used for early stopping
    
    score_name : Score name
    
    probs_fn : Function used to calculate probability of positive class from logits
    
    modelKwargs : kwargs passed to the model during initialisation
    
    trainKwargs : kwargs passed to the model during training
    
    testKwargs : kwargs passed to the model during testing for early stopping'''
    
    combined_metrics = []
    preNac_metrics = []
    
    for train_index,val_index in skf.split(train_df,train_df["pCR"]):
        model = Model_class(**modelKwargs)
        model = model.to(device)
        fold_train_df = train_df.iloc[train_index]
        fold_test_df = train_df.iloc[val_index]
        fold_train_dataset = ModelDataset(fold_train_df,DATASET_PATH,class_samples,loading_bar=False)
        fold_train_loader = DataLoader(fold_train_dataset,batch_size=batch_size,shuffle=True)
        fold_test_dataset = ModelDataset(fold_test_df,DATASET_PATH,loading_bar=False)
        fold_test_loader = DataLoader(fold_test_dataset,batch_size=batch_size)
        model,score = trainModel(model,
                                 train_loader=fold_train_loader,
                                 combine_timepoints=combine_timepoints,
                                 out_features=output_features,
                                 loss_fn = loss_fn,
                                 optimiser=optimiser_class(model.parameters()),
                                 num_epochs=num_epochs,
                                 val_loader=fold_test_loader,
                                 score_fn=score_fn,
                                 score_name=score_name,
                                 patience=num_epochs//4,
                                 trainKwargs=trainKwargs,
                                 testKwargs=testKwargs)
        y_true,y_score = probs_fn(model,fold_test_loader,combine_timepoints,output_features,testKwargs["mode"])
        print(f"Average Precision={(average_precision_score(y_true,y_score)):.4f}")
        combined_metrics.append(evaluate_model(model,
                                               fold_test_loader,
                                               combine_timepoints,
                                               output_features,
                                               mode="both"))
        preNac_metrics.append(evaluate_model(model,
                                               fold_test_loader,
                                               combine_timepoints,
                                               output_features,
                                               mode="preNac"))
        
        del model
        torch.cuda.empty_cache()
        
    return pd.DataFrame({"preNac":sum(preNac_metrics)/len(preNac_metrics),
            "Both":sum(combined_metrics)/len(combined_metrics),}).T.to_markdown()
        

## Model Tests

In [9]:
from models.CMC_Model.model import Model as CMC_Model

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.5445866499092306, 
'Average Precision': 0.38931456533232567, 
'Sensitivity / Recall / TPR': 0.7067204301075268, 
'Specificity / TNR': 0.4179383116883117, 
'PPV / Precision': 0.4787010990875746, 
'NPV': 0.8056943056943057, 
'Accuracy': 0.5174418604651163
'''


KeyboardInterrupt: 

In [ ]:
from models.CMC_SE.model import Model as CMC_SE_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.561273652422846, 
'Average Precision': 0.4331408019221398, 
'Sensitivity / Recall / TPR': 0.6610215053763441, 
'Specificity / TNR': 0.49261363636363636, 
'PPV / Precision': 0.43590604120695486, 
'NPV': 0.7934704184704184, 
'Accuracy': 0.5524281805745554
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=19.8124
ROC AUC=0.643452380952381
Epoch 1 Done. Average Loss=0.8168
ROC AUC=0.5589285714285714
Epoch 2 Done. Average Loss=0.7230
ROC AUC=0.556547619047619
Epoch 3 Done. Average Loss=0.5936
ROC AUC=0.505357142857143
Epoch 4 Done. Average Loss=0.5848
ROC AUC=0.5154761904761904
Epoch 5 Done. Average Loss=0.5707
ROC AUC=0.4622023809523809
Early stopping triggered at epoch 5. Best ROC AUC=0.643452380952381
Average Precision=0.4947
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=18.8987
ROC AUC=0.5325513196480939
Epoch 1 Done. Average Loss=0.6996
ROC AUC=0.4750733137829912
Epoch 2 Done. Average Loss=0.6752
ROC AUC=0.5120234604105572
Epoch 3 Done. Average Loss=0.6492
ROC AUC=0.5026392961876833
Epoch 4 Done. Average Loss=0.6648
ROC AUC=0.4574780058651026
Epoch 5 Done. Average Loss=0.6361
ROC AUC=0.5079178885630499
Early stopping triggered a

### CMC Model from benchmark paper with Squeeze-Excitation Blocks and Adaptive Average Pooling trained on preNac and postNac

In [ ]:
from models.CMC_SE_with_AvgPool import Model as CMC_SE_with_AP_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_with_AP_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)


#Ignore this
'''
'ROC_AUC': 0.6627363496718336, 
'Average Precision': 0.508258678020139, 
'Sensitivity / Recall / TPR': 0.660752688172043, 
'Specificity / TNR': 0.6693181818181818, 
'PPV / Precision': 0.5264654345102233, 
'NPV': 0.7861817427855164, 
'Accuracy': 0.6667920656634747
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7291
ROC AUC=0.6107142857142857
Epoch 1 Done. Average Loss=0.6929
ROC AUC=0.6205357142857143
Epoch 2 Done. Average Loss=0.6931
ROC AUC=0.5863095238095238
Epoch 3 Done. Average Loss=0.6925
ROC AUC=0.6595238095238095
Epoch 4 Done. Average Loss=0.6826
ROC AUC=0.6669642857142857
Epoch 5 Done. Average Loss=0.6895
ROC AUC=0.5889880952380953
Epoch 6 Done. Average Loss=0.6828
ROC AUC=0.6047619047619048
Epoch 7 Done. Average Loss=0.6803
ROC AUC=0.5779761904761904
Epoch 8 Done. Average Loss=0.6773
ROC AUC=0.5785714285714285
Epoch 9 Done. Average Loss=0.6848
ROC AUC=0.6407738095238096
Early stopping triggered at epoch 9. Best ROC AUC=0.6669642857142857
Average Precision=0.5273
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7379
ROC AUC=0.46041055718475077
Epoch 1 Done. Average Loss=0.6879
ROC AUC=0.47096774193548385
Epoch 2 Done. Average 

"\n'ROC_AUC': 0.6627363496718336, \n'Average Precision': 0.508258678020139, \n'Sensitivity / Recall / TPR': 0.660752688172043, \n'Specificity / TNR': 0.6693181818181818, \n'PPV / Precision': 0.5264654345102233, \n'NPV': 0.7861817427855164, \n'Accuracy': 0.6667920656634747\n"

|     |  ROC AUC| Average Precision Score| Sensitivity / Recall / TPR| Specificity / TNR| PPV / Precision|      NPV| Accuracy|
| --- | ------- | ---------------------- | ------------------------- | ---------------- | -------------- | ------- | ------- |
preNac| 0.614654|                0.471505|                   0.778226|          0.488636|         0.45382| 0.813777| 0.590595|
Both  | 0.587408|                0.452972|                   0.660753|          0.564854|         0.46069| 0.753904| 0.599179|


### CMC Model from benchmark paper with Squeeze-Excitation Blocks and Adaptive Average Pooling trained on preNac only

In [ ]:
from models.CMC_SE_with_AvgPool import Model as CMC_SE_with_AP_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_with_AP_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"preNac"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

### Dual ResNet Model trained on preNac and postNac
|     |  ROC AUC| Average Precision Score| Sensitivity / Recall / TPR| Specificity / TNR| PPV / Precision|      NPV| Accuracy|
| --- | ------- | ---------------------- | ------------------------- | ---------------- | -------------- | ------- | ------- |
preNac| 0.688885|                0.546715|                   0.825269|          0.515666|         0.50344| 0.862092| 0.626094|
Both  | 0.667877|                0.511665|                   0.741935|          0.583523|         0.53107| 0.826510| 0.640800|

In [ ]:
from models.ResNet.model import Model as DualResNetModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)

'''
         ROC AUC  Average Precision Score  Sensitivity / Recall / TPR  Specificity / TNR  PPV / Precision       NPV  Accuracy
preNac  0.688885                 0.546715                    0.825269           0.515666          0.50344  0.862092  0.626094
Both    0.667877                 0.511665                    0.741935           0.583523          0.53107  0.826510  0.640800
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7154
ROC AUC=0.6279761904761905
Epoch 1 Done. Average Loss=0.6890
ROC AUC=0.6077380952380952
Epoch 2 Done. Average Loss=0.6665
ROC AUC=0.7261904761904763
Epoch 3 Done. Average Loss=0.6606
ROC AUC=0.7238095238095239
Epoch 4 Done. Average Loss=0.6805
ROC AUC=0.7517857142857143
Epoch 5 Done. Average Loss=0.6430
ROC AUC=0.7440476190476191
Epoch 6 Done. Average Loss=0.6601
ROC AUC=0.7363095238095237
Epoch 7 Done. Average Loss=0.6627
ROC AUC=0.7119047619047619
Epoch 8 Done. Average Loss=0.6392
ROC AUC=0.7446428571428572
Epoch 9 Done. Average Loss=0.6536
ROC AUC=0.731845238095238
Early stopping triggered at epoch 9. Best ROC AUC=0.7517857142857143
Average Precision=0.6540
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6963
ROC AUC=0.5038123167155425
Epoch 1 Done. Average Loss=0.6407
ROC AUC=0.5378299120234604
Epoch 2 Done. Average Los

"\n'ROC_AUC': 0.6992595307917888, \n'Average Precision': 0.50554374036523, \n'Sensitivity / Recall / TPR': 0.7755376344086021, \n'Specificity / TNR': 0.5966720779220779, \n'PPV / Precision': 0.5266954663693795, \n'NPV': 0.8417085295656724, \n'Accuracy': 0.6609439124487004\n"

### Dual ResNet Model trained on preNac only
|     |  ROC AUC| Average Precision Score| Sensitivity / Recall / TPR| Specificity / TNR| PPV / Precision|      NPV| Accuracy|
| --- | ------- | ---------------------- | ------------------------- | ---------------- | -------------- | ------- | ------- |
preNac| 0.686923|                0.562943|                   0.751613|           0.60211|        0.531478| 0.817931| 0.655369|
Both  | 0.686777|                0.562903|                   0.751613|           0.60211|        0.531478| 0.817931| 0.655369|

In [ ]:
from models.ResNet.model import Model as DualResNetModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"preNac"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)
'''
         ROC AUC  Average Precision Score  Sensitivity / Recall / TPR  Specificity / TNR  PPV / Precision       NPV  Accuracy
preNac  0.686923                 0.562943                    0.751613            0.60211         0.531478  0.817931  0.655369
Both    0.686777                 0.562903                    0.751613            0.60211         0.531478  0.817931  0.655369
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7252
ROC AUC=0.5714285714285714
Epoch 1 Done. Average Loss=0.7093
ROC AUC=0.6410714285714286
Epoch 2 Done. Average Loss=0.6927
ROC AUC=0.6208333333333333
Epoch 3 Done. Average Loss=0.6803
ROC AUC=0.5636904761904762
Epoch 4 Done. Average Loss=0.6914
ROC AUC=0.6642857142857143
Epoch 5 Done. Average Loss=0.6573
ROC AUC=0.6375000000000001
Epoch 6 Done. Average Loss=0.6773
ROC AUC=0.6767857142857143
Epoch 7 Done. Average Loss=0.6880
ROC AUC=0.7166666666666667
Epoch 8 Done. Average Loss=0.6773
ROC AUC=0.6970238095238096
Epoch 9 Done. Average Loss=0.6952
ROC AUC=0.6526785714285714
Epoch 10 Done. Average Loss=0.6617
ROC AUC=0.6482142857142857
Epoch 11 Done. Average Loss=0.6659
ROC AUC=0.6535714285714286
Epoch 12 Done. Average Loss=0.6607
ROC AUC=0.4994047619047619
Early stopping triggered at epoch 12. Best ROC AUC=0.7166666666666667
Average Precision=0.6196
Dataset initialised with 346 entri

In [ ]:
from models.ResNet_SE.ResNet_SE_r_16 import Model as DualResNetSE_r_16_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_16_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.6598273634967183, 
'Average Precision': 0.4810334624219552, 
'Sensitivity / Recall / TPR': 0.7448924731182796, 
'Specificity / TNR': 0.592775974025974, 
'PPV / Precision': 0.5024766899766899, 
'NPV': 0.8146424349881797, 
'Accuracy': 0.6462380300957593
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7342
ROC AUC=0.6339285714285715
Epoch 1 Done. Average Loss=0.7007
ROC AUC=0.6702380952380953
Epoch 2 Done. Average Loss=0.6674
ROC AUC=0.6601190476190476
Epoch 3 Done. Average Loss=0.6771
ROC AUC=0.6267857142857143
Epoch 4 Done. Average Loss=0.6952
ROC AUC=0.6202380952380953
Epoch 5 Done. Average Loss=0.6656
ROC AUC=0.6702380952380952
Epoch 6 Done. Average Loss=0.6885
ROC AUC=0.6238095238095238
Early stopping triggered at epoch 6. Best ROC AUC=0.6702380952380953
Average Precision=0.4752
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7160
ROC AUC=0.5824046920821114
Epoch 1 Done. Average Loss=0.6543
ROC AUC=0.509090909090909
Epoch 2 Done. Average Loss=0.6513
ROC AUC=0.5378299120234603
Epoch 3 Done. Average Loss=0.6281
ROC AUC=0.5313782991202346
Epoch 4 Done. Average Loss=0.6558
ROC AUC=0.5020527859237536
Epoch 5 Done. Average Los

In [ ]:
from models.ResNet_SE.ResNet_SE_r_4 import Model as DualResNetSE_r_4_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_4_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})
print(four_fold_scores)

'''
'ROC_AUC': 0.691136712749616, 
'Average Precision': 0.5321514416865775, 
'Sensitivity / Recall / TPR': 0.7120967741935483, 
'Specificity / TNR': 0.6651785714285714, 
'PPV / Precision': 0.5407551766436784, 
'NPV': 0.8130799755799756, 
'Accuracy': 0.6814979480164158
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7067
ROC AUC=0.5910714285714286
Epoch 1 Done. Average Loss=0.6985
ROC AUC=0.6339285714285714
Epoch 2 Done. Average Loss=0.6738
ROC AUC=0.6642857142857143
Epoch 3 Done. Average Loss=0.6905
ROC AUC=0.6303571428571428
Epoch 4 Done. Average Loss=0.6419
ROC AUC=0.5630952380952381
Epoch 5 Done. Average Loss=0.6600
ROC AUC=0.7154761904761905
Epoch 6 Done. Average Loss=0.6958
ROC AUC=0.6821428571428572
Epoch 7 Done. Average Loss=0.6550
ROC AUC=0.6136904761904762
Epoch 8 Done. Average Loss=0.6433
ROC AUC=0.738095238095238
Epoch 9 Done. Average Loss=0.6573
ROC AUC=0.7226190476190476
Epoch 10 Done. Average Loss=0.6806
ROC AUC=0.6565476190476192
Epoch 11 Done. Average Loss=0.6716
ROC AUC=0.6607142857142857
Epoch 12 Done. Average Loss=0.6615
ROC AUC=0.6642857142857143
Epoch 13 Done. Average Loss=0.6483
ROC AUC=0.7047619047619048
Early stopping triggered at epoch 13. Best ROC AUC=0.738095238095238

### Conv Mixer 128/4 trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.681448 |                  0.497225 |                     0.851075 |            0.515828 |          0.498676 | 0.858586 |   0.63485  |
| Both   |  0.618113 |                  0.447283 |                     0.802688 |            0.470617 |          0.45785  | 0.831653 |   0.587688 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":4,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":64},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)


Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7183
ROC AUC=0.6386904761904761
Epoch 1 Done. Average Loss=0.6436
ROC AUC=0.7029761904761905
Epoch 2 Done. Average Loss=0.6103
ROC AUC=0.6886904761904762
Epoch 3 Done. Average Loss=0.6308
ROC AUC=0.5148809523809523
Epoch 4 Done. Average Loss=0.6105
ROC AUC=0.5523809523809524
Epoch 5 Done. Average Loss=0.5685
ROC AUC=0.675
Epoch 6 Done. Average Loss=0.5575
ROC AUC=0.6178571428571429
Early stopping triggered at epoch 6. Best ROC AUC=0.7029761904761905
Average Precision=0.4818
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6752
ROC AUC=0.5882697947214076
Epoch 1 Done. Average Loss=0.6249
ROC AUC=0.5255131964809384
Epoch 2 Done. Average Loss=0.6175
ROC AUC=0.5501466275659824
Epoch 3 Done. Average Loss=0.6096
ROC AUC=0.501466275659824
Epoch 4 Done. Average Loss=0.5772
ROC AUC=0.5595307917888563
Epoch 5 Done. Average Loss=0.5909
ROC 

### Conv Mixer 256/6 trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.678469 |                  0.544371 |                     0.702957 |            0.637662 |          0.524984 | 0.801245 |    0.66091 |
| Both   |  0.612599 |                  0.478796 |                     0.540054 |            0.733117 |          0.551455 | 0.761019 |    0.66368 

In [9]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":256,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7870
ROC AUC=0.594047619047619
Epoch 1 Done. Average Loss=0.7332
ROC AUC=0.5452380952380953
Epoch 2 Done. Average Loss=0.6439
ROC AUC=0.6761904761904762
Epoch 3 Done. Average Loss=0.6396
ROC AUC=0.7250000000000001
Epoch 4 Done. Average Loss=0.5938
ROC AUC=0.593452380952381
Epoch 5 Done. Average Loss=0.6657
ROC AUC=0.7101190476190475
Epoch 6 Done. Average Loss=0.6013
ROC AUC=0.6845238095238095
Epoch 7 Done. Average Loss=0.5995
ROC AUC=0.6976190476190477
Epoch 8 Done. Average Loss=0.6172
ROC AUC=0.6607142857142858
Early stopping triggered at epoch 8. Best ROC AUC=0.7250000000000001
Average Precision=0.5731
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7272
ROC AUC=0.5079178885630499
Epoch 1 Done. Average Loss=0.6631
ROC AUC=0.49266862170087977
Epoch 2 Done. Average Loss=0.6442
ROC AUC=0.5519061583577712
Epoch 3 Done. Average Los

### Conv Mixer 128/6 trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.695554 |                  0.558633 |                     0.785215 |            0.579383 |          0.51173  | 0.829545 |   0.652326 |
| Both   |  0.617869 |                  0.471568 |                     0.8      |            0.464286 |          0.482102 | 0.876818 |   0.584918 |

In [10]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7616
ROC AUC=0.5886904761904762
Epoch 1 Done. Average Loss=0.6953
ROC AUC=0.6351190476190476
Epoch 2 Done. Average Loss=0.6074
ROC AUC=0.7017857142857142
Epoch 3 Done. Average Loss=0.6474
ROC AUC=0.6720238095238095
Epoch 4 Done. Average Loss=0.6196
ROC AUC=0.6821428571428572
Epoch 5 Done. Average Loss=0.6155
ROC AUC=0.6595238095238095
Epoch 6 Done. Average Loss=0.6039
ROC AUC=0.6160714285714285
Epoch 7 Done. Average Loss=0.6392
ROC AUC=0.5910714285714286
Early stopping triggered at epoch 7. Best ROC AUC=0.7017857142857142
Average Precision=0.5746
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7500
ROC AUC=0.6052785923753665
Epoch 1 Done. Average Loss=0.6145
ROC AUC=0.4656891495601173
Epoch 2 Done. Average Loss=0.6329
ROC AUC=0.5378299120234604
Epoch 3 Done. Average Loss=0.6561
ROC AUC=0.5208211143695015
Epoch 4 Done. Average Lo

### Conv Mixer 128/6 trained on preNac only
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.645287 |                  0.51019  |                      0.79543 |            0.502192 |          0.463588 | 0.833445 |   0.605369 |
| Both   |  0.645436 |                  0.510238 |                      0.79543 |            0.502192 |          0.463588 | 0.833445 |   0.605369 |

In [11]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"preNac"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7151
ROC AUC=0.6053571428571429
Epoch 1 Done. Average Loss=0.7149
ROC AUC=0.6702380952380953
Epoch 2 Done. Average Loss=0.6808
ROC AUC=0.5136904761904761
Epoch 3 Done. Average Loss=0.6595
ROC AUC=0.5672619047619049
Epoch 4 Done. Average Loss=0.6617
ROC AUC=0.6029761904761904
Epoch 5 Done. Average Loss=0.6878
ROC AUC=0.5994047619047619
Epoch 6 Done. Average Loss=0.6576
ROC AUC=0.6636904761904762
Early stopping triggered at epoch 6. Best ROC AUC=0.6702380952380953
Average Precision=0.5699
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7579
ROC AUC=0.509090909090909
Epoch 1 Done. Average Loss=0.6276
ROC AUC=0.4868035190615836
Epoch 2 Done. Average Loss=0.6455
ROC AUC=0.4686217008797654
Epoch 3 Done. Average Loss=0.6301
ROC AUC=0.4633431085043988
Epoch 4 Done. Average Loss=0.6135
ROC AUC=0.42815249266862176
Epoch 5 Done. Average Lo

In [12]:
gc.collect()

80